
## Notebook to run sparklens on a cluster in python
---
In order to run this, you need to:
- build the sparklens jar by issuing 'sbt assembly' at a command prompt. this will build a jar file in target/scala-2.11/sparklens-assembly-0.1.jar
- add this jar file to a Volume and install it to the cluster you want to run it on 
- install the org.json4s:json4s-jackson_2.13:4.1.0-M8 also to the same cluster

In [0]:
val df = spark.read.format("csv").option("header", true).option("inferSchema", true).load("/Volumes/catadb360dev/schemaadb360dev/bronze/flights-1m.csv")

In [0]:
display(df.limit(25))

In [0]:
import org.apache.spark.sql.functions.{avg}

val aggregated_df = df.groupBy("DISTANCE", "AIR_TIME").agg(
    avg("DISTANCE").alias("avg_distance"),
    avg("AIR_TIME").alias("avg_airtime")
)


In [0]:
aggregated_df.write 
    .mode("overwrite") 
    .format("delta") 
    .saveAsTable("avgdistandair")

In [0]:
import com.qubole.sparklens.{QuboleNotebookListener}

val QNL = new QuboleNotebookListener(sc.getConf)
sc.addSparkListener(QNL)

In [0]:
QNL.profileIt {
  val df = spark.read.format("csv").option("header", true).option("inferSchema", true).load("/Volumes/catadb360dev/schemaadb360dev/bronze/flights-1m.csv")

val aggregated_df = df.groupBy("DISTANCE", "AIR_TIME").agg(
    avg("DISTANCE").alias("avg_distance"),
    avg("AIR_TIME").alias("avg_airtime")
)

aggregated_df.write 
    .mode("overwrite") 
    .format("delta") 
    .saveAsTable("avgdistandair")

}